# OS-A03 · Professional Obligations · executable scenario laboratory

**Required product:** professional judgement memorandum. Synthetic Kintaba training cases only. The notebook supports issue extraction, evidence checking, information screening, baseline and AI-assisted comparison. An authorized statistician makes and approves the decision. No network or API key is required. The deterministic mock assistant is a rehearsal; it is not a model evaluation or a substitute for approved AI service testing.

Open this notebook from any folder, including as a single file downloaded from the learning site. Synthetic training data are embedded. Outputs are written to an `outputs/` folder under the notebook kernel’s current working directory. Python 3.10+ standard library suffices. Execute cells in order.


In [ ]:
from pathlib import Path
import csv, json, time, hashlib, re
# Synthetic teaching data travel inside this notebook so a single-file website download works.
EMBEDDED_SCENARIOS = json.loads('[{"id": "KNT-PUB-01", "title": "Pressure to publish a misleading headline", "purpose": "Quarterly labour bulletin", "decision": "Whether an AI-written headline may be released", "owner": "Chief statistician", "environment": "Approved internal drafting service", "facts": "Estimate 12.4%, prior 12.1%; uncertainty interval overlaps; a frame change reduces comparability.", "protected": "none", "authority": "Release policy requires statistics sign-off; local clause must be checked.", "ai_draft": "Employment clearly deteriorated; publish immediately. The Statistical Release Act 2025 requires this headline.", "obligations": "impartiality|transparency|quality|accountability", "risk": "Fabricated authority; exaggerated change; omitted frame break.", "reference_disposition": "reject", "critical": "fabricated_authority|quality", "source_ids": "E01|E02|P01"}, {"id": "KNT-ADM-02", "title": "Administrative record coding", "purpose": "Code mortality narratives", "decision": "Whether external AI may process free text", "owner": "Data steward", "environment": "Public external service proposed", "facts": "Names removed; narratives retain rare places, occupations, and events; no vendor clearance.", "protected": "rare free text; possible reidentification", "authority": "Statistical secrecy and supplier clearance require local legal review.", "ai_draft": "Data are anonymous because names were removed. Upload all narratives; no review is needed.", "obligations": "confidentiality|quality|accountability", "risk": "Inferential disclosure; uncertain retention; false anonymization.", "reference_disposition": "reject", "critical": "confidentiality|authority", "source_ids": "E03|P02"}, {"id": "KNT-ACC-03", "title": "Accessible provisional indicator", "purpose": "Provide timely mortality indicator", "decision": "Whether provisional aggregate may be released", "owner": "Release committee", "environment": "Approved restricted environment", "facts": "Cause coding incomplete; secure processing approved; aggregate expert check pending.", "protected": "unit level records remain internal", "authority": "Provisional release policy exists; exact clause requires confirmation.", "ai_draft": "Publish a final indicator without uncertainty because users need speed.", "obligations": "transparency|quality|accountability|impartiality", "risk": "Provisional presented as final; missing review.", "reference_disposition": "condition", "critical": "quality|release", "source_ids": "E04|P03"}]')
EMBEDDED_EVIDENCE = json.loads('[{"id": "E01", "kind": "synthetic_evidence", "claim": "Current estimate and interval", "status": "verified_for_training", "source": "Kintaba synthetic bulletin", "public": "yes"}, {"id": "E02", "kind": "synthetic_evidence", "claim": "Frame break affects comparison", "status": "verified_for_training", "source": "Kintaba synthetic methods note", "public": "yes"}, {"id": "E03", "kind": "synthetic_evidence", "claim": "Rare narrative identifiers persist", "status": "verified_for_training", "source": "Kintaba synthetic risk note", "public": "no"}, {"id": "E04", "kind": "synthetic_evidence", "claim": "Cause coding incomplete", "status": "verified_for_training", "source": "Kintaba synthetic situation report", "public": "yes"}, {"id": "P01", "kind": "local_policy_placeholder", "claim": "Release sign-off clause", "status": "unverified_localization_required", "source": "Institutional policy to be supplied", "public": "yes"}, {"id": "P02", "kind": "local_policy_placeholder", "claim": "Secrecy and vendor clause", "status": "unverified_localization_required", "source": "Institutional policy to be supplied", "public": "no"}, {"id": "P03", "kind": "local_policy_placeholder", "claim": "Provisional release clause", "status": "unverified_localization_required", "source": "Institutional policy to be supplied", "public": "yes"}]')
ROOT = Path.cwd()  # output location; no data-file assumption
rows = EMBEDDED_SCENARIOS
evidence = {r['id']: r for r in EMBEDDED_EVIDENCE}
assert len(rows) == 3 and all(set(r['source_ids'].split('|')) <= evidence.keys() for r in rows)
# Where package data are present, detect mismatches instead of silently mixing versions.
if (ROOT/'data/scenarios.csv').exists() and (ROOT/'data/evidence_register.csv').exists():
    disk_rows = list(csv.DictReader((ROOT/'data/scenarios.csv').open(encoding='utf-8', newline='')))
    disk_evidence = list(csv.DictReader((ROOT/'data/evidence_register.csv').open(encoding='utf-8', newline='')))
    assert disk_rows == rows and disk_evidence == EMBEDDED_EVIDENCE, 'Package data differ from embedded version; use a matching package.'
print('Loaded', len(rows), 'synthetic cases and', len(evidence), 'evidence entries; output folder:', ROOT/'outputs')


## 1. Task, authority and common acceptance standard

Record intended use, accountable owner and actual local authority before deployment. P01–P03 are **unverified placeholders**. The common standard below applies to baseline and assisted analyses. Critical failures override any score.


In [ ]:
case = rows[0]  # change to rows[1] or rows[2] and rerun downstream cells
acceptance = {'authority_trace': True, 'evidence_trace': True, 'information_screen': True, 'five_obligations_considered': True, 'counterargument': True, 'human_owner': True, 'safeguards_and_review': True}
assert case['owner'] and case['decision']
print(case['id'],case['title'],'| owner:',case['owner'])
print('Unverified local authority:',case['authority'])


## 2. Conventional baseline

Do the work before seeing the mock assistant's analysis. Edit the baseline record with your own notes and actual measured minutes; the example values are illustrative and cannot establish efficiency evidence.


In [ ]:
baseline = {'facts': case['facts'], 'obligations': case['obligations'].split('|'), 'options':['accept','condition','restrict','reject','escalate'], 'decision':'', 'rationale':'', 'confidence':'', 'minutes':None, 'review_minutes':None, 'rework_minutes':None}
# Learner entry: set decision, rationale, confidence, minutes, review_minutes and rework_minutes.
assert baseline['minutes'] is None  # no fabricated timing is inserted
print('Baseline template prepared; enter actual work and time before continuing.')


## 3. Information classification and approved input

Apply data classification before any AI interaction. Only the allowlisted scenario summary crosses the simulated assistant boundary. Protected content and policy placeholders are withheld. In real work, confirm service approval, logging, retention and access before interaction.


In [ ]:
ALLOW = ('id','title','purpose','decision','owner','environment','facts','obligations','risk')
withheld = {k:case[k] for k in ('protected','authority','source_ids','ai_draft','critical')}
permitted = {k:case[k] for k in ALLOW}
assert all(k not in permitted for k in withheld)
assert case['protected'] not in json.dumps(permitted)
assert case['ai_draft'] not in json.dumps(permitted)
classification = {'permitted_fields':list(ALLOW),'excluded_fields':list(withheld),'reason':'Protected narrative, asserted policy and adversarial draft stay within controlled human verification.'}
print(json.dumps(classification,indent=2))


## 4. Bounded AI task specification and offline simulation

This task requests issues, counterarguments and a draft structure. It forbids invented authority and final judgement. The built-in simulator is deterministic, local and does not ingest protected fields. To use an institutionally approved AI tool, a human must first clear the environment and retain a controlled interaction record.


In [ ]:
task_spec = {'purpose':case['purpose'], 'input':permitted, 'requested':['identify five obligation issues','name missing evidence','offer counterarguments','draft memorandum headings'], 'constraints':['No invented law or policy','No protected information','Do not make or approve final decision','Cite only supplied source IDs after independent lookup','Say unknown when evidence is absent']}
def offline_assistant(spec):
    item=spec['input']
    return {'case_id':item['id'],'issues':item['obligations'].split('|'),'risks':[x.strip() for x in item['risk'].split(';')], 'counterarguments':['Could a narrower approved workflow retain public value?','What happens if action is delayed?'], 'missing_evidence':['Exact local policy clause and authorized sign-off','Independent confirmation of material statistical claims'], 'draft_headings':['Decision requested','Verified facts','Authority','Information handling','Obligations','Options','Verification','Judgement','Safeguards','Review','Efficiency'], 'disposition':'HUMAN DECISION REQUIRED'}
start=time.perf_counter(); suggestion=offline_assistant(task_spec); simulation_seconds=time.perf_counter()-start
assert suggestion['disposition']=='HUMAN DECISION REQUIRED'
print(json.dumps(suggestion,indent=2)); print('Simulation runtime seconds:',round(simulation_seconds,6))


## 5. Independent source and adversarial verification

The supplied `ai_draft` deliberately contains unsafe claims. Detect unsupported authority, quality problems and disclosure hazards. Do not treat a synthetic reference disposition as governing a real institution.


In [ ]:
linked=[evidence[k] for k in case['source_ids'].split('|')]
unchecked=[r['id'] for r in linked if r['status']!='verified_for_training']
challenge={'case_id':case['id'],'draft_claims':case['ai_draft'],'unverified_sources':unchecked,'critical_risks':case['critical'].split('|'),'contradictions':case['risk'],'human_checks':['Locate authoritative local clause','Check source figures and uncertainty','Confirm information handling and release authority','Challenge omission and counterarguments']}
assert unchecked and all(evidence[k]['kind']=='local_policy_placeholder' for k in unchecked)
print(json.dumps(challenge,indent=2))


## 6. Human judgement memorandum

Fill the fields with accountable reasoning. Blank fields remain blank rather than being supplied by the simulator. Choose accept, condition, restrict, reject or escalate; record owners, due dates, stop rule and correction route.


In [ ]:
memo={'case_id':case['id'],'decision_requested':case['decision'],'decision_owner':case['owner'],'verified_facts':'','source_ids':case['source_ids'].split('|'),'authority_status':'LOCAL VERIFICATION REQUIRED','information_handling':classification,'obligation_analysis':{x:'' for x in ('impartiality','confidentiality','transparency','quality','accountability')},'options_and_nonaction':'','verification_and_corrections':'','disposition':'','rationale':'','safeguards':[],'review_date':'','stopping_rule':'','correction_route':'','limitations':'Synthetic training only; local authority unverified.'}
allowed={'accept','condition','restrict','reject','escalate'}
def assess(m):
    missing=[k for k in ('verified_facts','options_and_nonaction','verification_and_corrections','disposition','rationale','review_date','stopping_rule','correction_route') if not m[k]]
    missing += ['obligation:'+k for k,v in m['obligation_analysis'].items() if not v]
    if m['disposition'] not in allowed: missing.append('valid human disposition')
    if not m['safeguards']: missing.append('safeguards with owner, due date and evidence')
    if m['authority_status']=='LOCAL VERIFICATION REQUIRED': missing.append('verified local authority / escalation')
    return {'ready_for_approval':not missing,'missing':missing}
print(json.dumps(assess(memo),indent=2))


## 7. Same-standard quality and quality-adjusted efficiency

Record measured active time for both routes including verification and rework. Only claim positive savings if each route passes the **same** acceptance standard and no critical failure is present. This comparison cannot infer real AI productivity from the offline simulator.


In [ ]:
assisted={'generation_minutes':None,'verification_minutes':None,'rework_minutes':None,'quality_checks':{},'critical_failures':[]}; baseline['quality_checks']={};baseline['critical_failures']=[]
def compare(b,a,required):
    values=[b['minutes'],b['review_minutes'],b['rework_minutes'],a['generation_minutes'],a['verification_minutes'],a['rework_minutes']]
    if any(x is None for x in values): return {'claim_allowed':False,'reason':'Measured time missing','net_minutes':None}
    if any(not isinstance(x,(int,float)) or isinstance(x,bool) or x<0 for x in values): raise ValueError('Minutes must be nonnegative numbers')
    pass_b=all(b['quality_checks'].get(k) is True for k in required) and not b['critical_failures']
    pass_a=all(a['quality_checks'].get(k) is True for k in required) and not a['critical_failures']
    delta=sum(values[:3])-sum(values[3:])
    return {'baseline_pass':pass_b,'assisted_pass':pass_a,'net_minutes':delta,'claim_allowed':pass_b and pass_a and delta>0,'reason':'Same-standard pass required; include all verification and rework'}
comparison=compare(baseline,assisted,acceptance)
print(json.dumps(comparison,indent=2))


## 8. Reproducible guardrail checks and controlled export

These tests verify input exclusion, missing authority, stop conditions and the time formula. They do not assess the quality of your human rationale. Export is clearly marked as a candidate until all approval fields and local authority are verified.


In [ ]:
assert not assess(memo)['ready_for_approval']
test_b={'minutes':40,'review_minutes':5,'rework_minutes':0,'quality_checks':dict(acceptance),'critical_failures':[]}
test_a={'generation_minutes':8,'verification_minutes':12,'rework_minutes':5,'quality_checks':dict(acceptance),'critical_failures':[]}
assert compare(test_b,test_a,acceptance)['net_minutes']==20
assert compare(test_b,test_a,acceptance)['claim_allowed']
test_a['critical_failures']=['fabricated authority']; assert not compare(test_b,test_a,acceptance)['claim_allowed']
test_a['critical_failures']=[];test_a['quality_checks']['authority_trace']=False;assert not compare(test_b,test_a,acceptance)['claim_allowed']
assert all(k not in task_spec['input'] for k in withheld)
export={'scenario':case['id'],'synthetic':True,'task_specification':task_spec,'classification':classification,'assistant_output':suggestion,'challenge':challenge,'memo':memo,'memo_assessment':assess(memo),'baseline':baseline,'assisted':assisted,'comparison':comparison,'simulation_seconds':simulation_seconds}
(ROOT/'outputs').mkdir(exist_ok=True)
out=ROOT/'outputs'/f"{case['id']}_CANDIDATE.json";out.write_text(json.dumps(export,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print('Guardrail tests passed; wrote candidate:',out)
